# Apache Spark Overview

## Spark
Spark is up to 100 times faster than traditional MapReduce, offering a powerful platform for distributed data processing and analytics.

## Spark Components

### How Spark Works?
In Apache Spark, components are organized around the Driver and Executors.

- **Driver & Executors:**
  - **Driver:** The master node that manages the lifecycle of the Spark application, breaks down tasks, coordinates work, and collects results.
  - **Executors:** Worker nodes that execute tasks assigned by the Driver, performing computations and storing data for each partition.

- **Job Division in Stages:**
  - A Spark job is divided into stages, each representing a step in the job.
  - **Narrow Transformations** (e.g., `map`, `filter`): Operate on a single partition without data exchange between nodes.
  - **Wide Transformations** (e.g., `reduceByKey`, `groupByKey`): Require shuffling and split stages, causing dependencies across partitions.

- **Shuffle:**
  - The process of redistributing data across nodes during wide transformations, involving disk I/O and network transfers, which can affect performance.

These components work together to optimize parallel processing and data distribution in a Spark application.

### Key Concepts

- **Partition:**
  - A logical chunk of data within a Spark RDD or DataFrame, enabling parallel processing across the cluster. Efficient partitioning helps balance the workload.

- **Transformations:**
  - Operations on RDDs or DataFrames (like `map`, `filter`, `flatMap`) that define a new dataset from an existing one. They are **lazy** and executed only when an action is invoked.

- **Actions:**
  - Actions trigger execution, producing results (e.g., `collect`, `count`, `saveAsTextFile`) and returning data to the Driver or writing output.

- **Lazy Evaluation:**
  - Spark builds a lineage of operations, executing them only upon an action. This allows optimizations like pipelining and minimizing data shuffles.

## Core Elements

- **Spark Session:**
  - The unified entry point to Spark functionality, managing configurations, DataFrame API, and SQL operations, replacing `SparkContext` and `SQLContext` in Spark 2.0.

- **Spark DataFrames:**
  - A distributed collection of data organized into named columns, like a table in a database, allowing SQL-like operations on structured and semi-structured data.

  - **Example: Creating a DataFrame from CSV**
    ```python
    df = spark.read.csv("path/to/file.csv", header=True, inferSchema=True)
    df.show()
    ```
    This loads data into a DataFrame `df` for SQL and DataFrame operations (like `filter`, `select`, and `groupBy`).

- **Data Partitioning in DataFrames:**
  - Data in a DataFrame is divided into partitions across nodes, each undergoing transformations independently to enable parallel processing.

## Spark Execution Planning

- **Logical Execution Planning:**
  - Spark builds a logical plan for the job, outlining operations without concern for data movement or resources. Spark then optimizes this plan for efficient execution.

- **Physical Execution Planning:**
  - Converts the logical plan into stages and tasks, considering data locality and resource availability, and optimizing partitioning, caching, and shuffling.

- **DAG (Directed Acyclic Graph):**
  - A DAG in Spark represents the sequence of operations required to complete a job, with each stage as a set of transformations and edges as data flow dependencies. The DAG helps Spark break down jobs into stages and optimize execution.


In [83]:
import findspark
findspark.init()

In [84]:
import pyspark
from pyspark.sql import SparkSession

In [85]:
spark=SparkSession.builder.getOrCreate()
# spark=SparkSession.builder.appName("learn").master("local[*]").getOrCreate()
spark

In [86]:
type(spark)

pyspark.sql.session.SparkSession

In [87]:
# help(spark.createDataFrame)

In [88]:
data=[(1,"Nishant"),(2,"ketu")]
df1=spark.createDataFrame(data=data,schema=['id','name'])
df1.show()

+---+-------+
| id|   name|
+---+-------+
|  1|Nishant|
|  2|   ketu|
+---+-------+



In [89]:
df1.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)



In [90]:
from pyspark.sql.types import *
schema=StructType([StructField(name="id",dataType=IntegerType()),
            StructField(name="name",dataType=StringType())])

In [91]:
type(schema)

pyspark.sql.types.StructType

In [92]:
df2=spark.createDataFrame(data=data,schema=schema)
df2.show()

+---+-------+
| id|   name|
+---+-------+
|  1|Nishant|
|  2|   ketu|
+---+-------+



In [93]:
df2.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)



In [94]:
data=[{"id":1,"name":"nishant"},
      {"id":2,"name":"ketu"}
     ]
df3=spark.createDataFrame(data=data)
df3.show()

+---+-------+
| id|   name|
+---+-------+
|  1|nishant|
|  2|   ketu|
+---+-------+



In [95]:
#Reading CSV

In [96]:
df1=spark.read.csv(path='people-100.csv',header=True,inferSchema=True)
# df1=spark.read.csv(path='people-100.csv',header=True)
# without inferSchema everything is string
# use array of path for adding multiple files
# df=spark.read.csv(['people-100.csv','people-100-2.csv'],header=True,inferSchema=True)
# df=spark.read.csv('./csvfile/',header=True,inferSchema=True)
display(df1)
df1.show()

DataFrame[Index: int, User Id: string, First Name: string, Last Name: string, Sex: string, Email: string, Phone: string, Date of birth: date, Job Title: string]

+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+--------------------+
|Index|        User Id|First Name|Last Name|   Sex|               Email|               Phone|Date of birth|           Job Title|
+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+--------------------+
|    1|88F7B33d2bcf9f5|    Shelby|  Terrell|  Male|elijah57@example.net|001-084-906-7849x...|   1945-10-26|     Games developer|
|    2|f90cD3E76f1A9b9|   Phillip|  Summers|Female|bethany14@example...|   214.112.6044x4913|   1910-03-24|      Phytotherapist|
|    3|DbeAb8CcdfeFC2c|  Kristine|   Travis|  Male|bthompson@example...|        277.609.7938|   1992-07-02|           Homeopath|
|    4|A31Bee3c201ef58|   Yesenia| Martinez|  Male|kaitlinkaiser@exa...|        584.094.6111|   2017-08-03|   Market researcher|
|    5|1bA7A3dc874da3c|      Lori|     Todd|  Male|buchananmanuel@ex...|   689-207-3558x7233|   1

In [97]:
# get number of partitions pyspark is creating
df1.rdd.getNumPartitions()
df1.printSchema()

root
 |-- Index: integer (nullable = true)
 |-- User Id: string (nullable = true)
 |-- First Name: string (nullable = true)
 |-- Last Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Date of birth: date (nullable = true)
 |-- Job Title: string (nullable = true)



In [98]:
from pyspark.sql.types import *

schema= StructType().add(field="Index",data_type=IntegerType())\
                    .add(field="User Id",data_type=StringType())\
                    .add(field="First Name",data_type=StringType())\
                    .add(field="Last Name",data_type=StringType())\
                    .add(field="Sex",data_type=StringType())\
                    .add(field="Email",data_type=StringType())\
                    .add(field="Phone",data_type=StringType())\
                    .add(field="Date of birth",data_type=DateType())\
                    .add(field="Job Title",data_type=StringType())


df2=spark.read.csv(path='people-100.csv',schema=schema,header=True)
display(df2)
df2.show(2)

DataFrame[Index: int, User Id: string, First Name: string, Last Name: string, Sex: string, Email: string, Phone: string, Date of birth: date, Job Title: string]

+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
|Index|        User Id|First Name|Last Name|   Sex|               Email|               Phone|Date of birth|      Job Title|
+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
|    1|88F7B33d2bcf9f5|    Shelby|  Terrell|  Male|elijah57@example.net|001-084-906-7849x...|   1945-10-26|Games developer|
|    2|f90cD3E76f1A9b9|   Phillip|  Summers|Female|bethany14@example...|   214.112.6044x4913|   1910-03-24| Phytotherapist|
+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
only showing top 2 rows



In [99]:
#JSON files
# in json we use multiline, if multiline json is there, By default its False
from pyspark.sql.types import *

# Define the schema for the new DataFrame
schema2 = StructType().add("Id", LongType()).add("Name", StringType()).add("Age", IntegerType()).add("Email", StringType()).add("City", StringType()).add("Food", ArrayType(StringType()))
df_json=spark.read.json(path='people.json',multiLine=True)
display(df_json)
df_json.show()

#df_json=spark.read.json('./someFolder/*.json')
# This can help to take all json from a folder

DataFrame[age: bigint, city: string, email: string, food: array<string>, id: bigint, name: string]

+---+------------+-------------------+--------------------+---+-------+
|age|        city|              email|                food| id|   name|
+---+------------+-------------------+--------------------+---+-------+
| 30| Los Angeles|  alice@example.com|     [apple, banana]|  1|  Alice|
| 28|    New York|    bob@example.com|[orange, strawberry]|  2|    Bob|
| 32|     Chicago|charlie@example.com|  [grape, blueberry]|  3|Charlie|
| 29|     Houston|  david@example.com|   [pineapple, kiwi]|  4|  David|
| 27|     Phoenix|    eve@example.com|      [mango, peach]|  5|    Eve|
| 31|Philadelphia|  frank@example.com|  [pear, watermelon]|  6|  Frank|
| 26| San Antonio|  grace@example.com|     [apple, cherry]|  7|  Grace|
| 34|   San Diego|  henry@example.com|[banana, grapefruit]|  8|  Henry|
| 25|      Dallas| isabel@example.com| [pineapple, papaya]|  9| Isabel|
| 33|    San Jose|   jack@example.com|      [orange, plum]| 10|   Jack|
+---+------------+-------------------+--------------------+---+-

In [100]:
# pyspark saves df data in chunks, not in a single file
# mode can we ignore , error , append , overwrite
df_json.write.json("save_json.json",mode='overwrite')

In [101]:
df_new_json=spark.read.json("save_json.json")
df_new_json.show()

+---+------------+-------------------+--------------------+---+-------+
|age|        city|              email|                food| id|   name|
+---+------------+-------------------+--------------------+---+-------+
| 30| Los Angeles|  alice@example.com|     [apple, banana]|  1|  Alice|
| 28|    New York|    bob@example.com|[orange, strawberry]|  2|    Bob|
| 32|     Chicago|charlie@example.com|  [grape, blueberry]|  3|Charlie|
| 29|     Houston|  david@example.com|   [pineapple, kiwi]|  4|  David|
| 27|     Phoenix|    eve@example.com|      [mango, peach]|  5|    Eve|
| 31|Philadelphia|  frank@example.com|  [pear, watermelon]|  6|  Frank|
| 26| San Antonio|  grace@example.com|     [apple, cherry]|  7|  Grace|
| 34|   San Diego|  henry@example.com|[banana, grapefruit]|  8|  Henry|
| 25|      Dallas| isabel@example.com| [pineapple, papaya]|  9| Isabel|
| 33|    San Jose|   jack@example.com|      [orange, plum]| 10|   Jack|
+---+------------+-------------------+--------------------+---+-

In [102]:
csv_df=spark.read.csv(path='people-100.csv',header=True)

In [103]:
# show function
# default 20 rows and letter in columns
# truncate is True by default, not show full columns like email here
# vertical will change table to vertical form.
# to limit data we have limit, removes data

csv_df.show(3, truncate=False)
# csv_df.show(5, truncate=5,vertical=True)
csv_df.limit(2).show()

+-----+---------------+----------+---------+------+---------------------+----------------------+-------------+---------------+
|Index|User Id        |First Name|Last Name|Sex   |Email                |Phone                 |Date of birth|Job Title      |
+-----+---------------+----------+---------+------+---------------------+----------------------+-------------+---------------+
|1    |88F7B33d2bcf9f5|Shelby    |Terrell  |Male  |elijah57@example.net |001-084-906-7849x73518|1945-10-26   |Games developer|
|2    |f90cD3E76f1A9b9|Phillip   |Summers  |Female|bethany14@example.com|214.112.6044x4913     |1910-03-24   |Phytotherapist |
|3    |DbeAb8CcdfeFC2c|Kristine  |Travis   |Male  |bthompson@example.com|277.609.7938          |1992-07-02   |Homeopath      |
+-----+---------------+----------+---------+------+---------------------+----------------------+-------------+---------------+
only showing top 3 rows

+-----+---------------+----------+---------+------+--------------------+--------------

In [104]:
csv_df.printSchema()

root
 |-- Index: string (nullable = true)
 |-- User Id: string (nullable = true)
 |-- First Name: string (nullable = true)
 |-- Last Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Date of birth: string (nullable = true)
 |-- Job Title: string (nullable = true)



In [105]:
# To call a column
# we can use 4 ways
# col('column')
# df.column
# df['column']
# expr('column')

# to select column
from pyspark.sql.functions import col,expr

csv_df.select(col("Index"),csv_df.Email,csv_df["Phone"],expr('sex')).show(2)


+-----+--------------------+--------------------+------+
|Index|               Email|               Phone|   sex|
+-----+--------------------+--------------------+------+
|    1|elijah57@example.net|001-084-906-7849x...|  Male|
|    2|bethany14@example...|   214.112.6044x4913|Female|
+-----+--------------------+--------------------+------+
only showing top 2 rows



In [106]:
# withColumn , drop , adding multiple column withColumns

from pyspark.sql.functions import col,lit

# casting,updating the colums
df1=csv_df.withColumn(colName='Index',col=col('Index').cast('Integer'))
df1.show(2)
# df1.printSchema()
df2=df1.withColumn(colName='Index',col=col("index")*5)
df2.show(2)

# creating new column, just give new name of column
df3=df2.withColumn(colName="country",col=lit("India"))
df3.show(2)
# df3.printSchema()

# to remove a colums
df3.drop("country").show(2)

# to add multiple columns
col_dict={"tax":lit(0),
          "fruit":lit("apple")}

df3.withColumns(col_dict)

+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
|Index|        User Id|First Name|Last Name|   Sex|               Email|               Phone|Date of birth|      Job Title|
+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
|    1|88F7B33d2bcf9f5|    Shelby|  Terrell|  Male|elijah57@example.net|001-084-906-7849x...|   1945-10-26|Games developer|
|    2|f90cD3E76f1A9b9|   Phillip|  Summers|Female|bethany14@example...|   214.112.6044x4913|   1910-03-24| Phytotherapist|
+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
only showing top 2 rows

+-----+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
|Index|        User Id|First Name|Last Name|   Sex|               Email|               Phone|Date of birth|

DataFrame[Index: int, User Id: string, First Name: string, Last Name: string, Sex: string, Email: string, Phone: string, Date of birth: string, Job Title: string, country: string, tax: int, fruit: string]

In [107]:
# withcolumRenamed
csv_df.withColumnRenamed('Index',"my_index").show(2)


+--------+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
|my_index|        User Id|First Name|Last Name|   Sex|               Email|               Phone|Date of birth|      Job Title|
+--------+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
|       1|88F7B33d2bcf9f5|    Shelby|  Terrell|  Male|elijah57@example.net|001-084-906-7849x...|   1945-10-26|Games developer|
|       2|f90cD3E76f1A9b9|   Phillip|  Summers|Female|bethany14@example...|   214.112.6044x4913|   1910-03-24| Phytotherapist|
+--------+---------------+----------+---------+------+--------------------+--------------------+-------------+---------------+
only showing top 2 rows



In [108]:
# struct type and struct field
# structtype structure StructType([StructField(name,dataType,nullabe)]), nullabe can be true or false

from pyspark.sql.types import *
data=[(1,("Nishant","ketu")),(2,("ajay","anand"))]
structName=StructType([StructField(name="first_name",dataType=StringType()),StructField(name="last_name",dataType=StringType())])
schema=StructType([StructField(name='id',dataType=IntegerType()),StructField(name='name',dataType=structName)])
df1=spark.createDataFrame(data,schema=schema)
# print beautifully
display(df1)
df1.show()

DataFrame[id: int, name: struct<first_name:string,last_name:string>]

+---+---------------+
| id|           name|
+---+---------------+
|  1|{Nishant, ketu}|
|  2|  {ajay, anand}|
+---+---------------+



In [109]:
 # string to schema , also we can do json to schema

schema_str = "name string , age int "
from pyspark.sql.types import _parse_datatype_string
spark_schema=_parse_datatype_string(schema_str)
spark_schema

StructType([StructField('name', StringType(), True), StructField('age', IntegerType(), True)])

In [110]:
#Array type
from pyspark.sql.types import *
data=[("nishant",[1,2,3,4,5]),("ketu",[1,4,3,4])]
schema=StructType([StructField("name",StringType()),StructField("my-num",ArrayType(IntegerType()))])
df1=spark.createDataFrame(data,schema)
df1=df1.withColumn("firstnum",col('my-num')[0])
df1.show()
df1.printSchema()

+-------+---------------+--------+
|   name|         my-num|firstnum|
+-------+---------------+--------+
|nishant|[1, 2, 3, 4, 5]|       1|
|   ketu|   [1, 4, 3, 4]|       1|
+-------+---------------+--------+

root
 |-- name: string (nullable = true)
 |-- my-num: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- firstnum: integer (nullable = true)



In [111]:
from pyspark.sql.functions import array
df2=df1.withColumn("merged_column",array(df1.firstnum,col('name')))
df2.show()

+-------+---------------+--------+-------------+
|   name|         my-num|firstnum|merged_column|
+-------+---------------+--------+-------------+
|nishant|[1, 2, 3, 4, 5]|       1| [1, nishant]|
|   ketu|   [1, 4, 3, 4]|       1|    [1, ketu]|
+-------+---------------+--------+-------------+



In [112]:
# explode ,it explore for every value of a array
# split  ,will split string into array
# Array_contain, will check an element in array

from pyspark.sql.functions import explode,col,split,array_contains

data=[("nishant",[6,7,3,4,5],'python,java'),("ketu",[8,9,3,4],'sql,dsa')]
df=spark.createDataFrame(data,['name','marks','skills'])
df.show()

# df.printSchema()
df1=df.withColumn('mark',explode(col('marks')))
df1.show()

df1=df.withColumn('skill',split(col('skills'),','))
df1.show()

df2=df1.withColumn('HaveDsa',array_contains(df1.skill,'dsa'))
df2.show()


+-------+---------------+-----------+
|   name|          marks|     skills|
+-------+---------------+-----------+
|nishant|[6, 7, 3, 4, 5]|python,java|
|   ketu|   [8, 9, 3, 4]|    sql,dsa|
+-------+---------------+-----------+

+-------+---------------+-----------+----+
|   name|          marks|     skills|mark|
+-------+---------------+-----------+----+
|nishant|[6, 7, 3, 4, 5]|python,java|   6|
|nishant|[6, 7, 3, 4, 5]|python,java|   7|
|nishant|[6, 7, 3, 4, 5]|python,java|   3|
|nishant|[6, 7, 3, 4, 5]|python,java|   4|
|nishant|[6, 7, 3, 4, 5]|python,java|   5|
|   ketu|   [8, 9, 3, 4]|    sql,dsa|   8|
|   ketu|   [8, 9, 3, 4]|    sql,dsa|   9|
|   ketu|   [8, 9, 3, 4]|    sql,dsa|   3|
|   ketu|   [8, 9, 3, 4]|    sql,dsa|   4|
+-------+---------------+-----------+----+

+-------+---------------+-----------+--------------+
|   name|          marks|     skills|         skill|
+-------+---------------+-----------+--------------+
|nishant|[6, 7, 3, 4, 5]|python,java|[python, java]|

In [113]:
# mapType, like dict in python
# can create schema of mapType like MapType(StringType(),StringType())

data=[('ketu',{'color':'red','age':5}),('nishant',{'color':'blue','age':12})]
df=spark.createDataFrame(data,['name','specs'])
df.show(truncate=False)
df.printSchema()

df1=df.withColumn('age',col('specs')['age'])
df1.show(truncate=False)

+-------+--------------------------+
|name   |specs                     |
+-------+--------------------------+
|ketu   |{color -> red, age -> 5}  |
|nishant|{color -> blue, age -> 12}|
+-------+--------------------------+

root
 |-- name: string (nullable = true)
 |-- specs: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)

+-------+--------------------------+---+
|name   |specs                     |age|
+-------+--------------------------+---+
|ketu   |{color -> red, age -> 5}  |5  |
|nishant|{color -> blue, age -> 12}|12 |
+-------+--------------------------+---+



In [114]:
# explode for Map will make a key and val column
from pyspark.sql.functions import explode,map_keys,map_values

# Explode the 'specs' map into separate rows
df2 = df.select("name", explode(col("specs")).alias("spec_key", "spec_value"))
# Show result
# df2.show()

# mapkeys and map_values are similar
df.withColumn('k',map_values(df.specs)).show()


+-------+--------------------+----------+
|   name|               specs|         k|
+-------+--------------------+----------+
|   ketu|{color -> red, ag...|  [red, 5]|
|nishant|{color -> blue, a...|[blue, 12]|
+-------+--------------------+----------+



# Row class

In [115]:
# Row is a class and we can also make object
# Row object representing a single record in a DataFrame, similar to a row in a table. 
# Rows store data in fields that can be accessed by index or by column name.
# Creation: Automatically created by Spark when using createDataFrame().
# Field Access: Access data by name (row['column_name']) or index (row[index]).

from pyspark.sql import Row
# row = Row("nishant",1)
row = Row(name="nishant",Class=1)

row['name']
row.name
print(row)
row[0]


Row(name='nishant', Class=1)


'nishant'

In [116]:
# it is a row object, we can make df from this

row1=Row(name="pawan",Class=1)
row2=Row(name="mandal",Class=4)
row3=Row(name="juhi",Class=5)

df=spark.createDataFrame([row1,row2,row3])
df.show()

# we can also create a object form Row and use it
title=Row('name','roll')
row1=title('ramesh',22)
row2=title('suresh',87)
spark.createDataFrame([row1,row2]).show()

# we can also use row as nested data
data=[Row(name="mohit",prop=Row(color='black',age=77)),Row(name="sumit",prop=Row(color='purple',age=97))]
df=spark.createDataFrame(data)
df.show()
df.printSchema()


+------+-----+
|  name|Class|
+------+-----+
| pawan|    1|
|mandal|    4|
|  juhi|    5|
+------+-----+

+------+----+
|  name|roll|
+------+----+
|ramesh|  22|
|suresh|  87|
+------+----+

+-----+------------+
| name|        prop|
+-----+------------+
|mohit| {black, 77}|
|sumit|{purple, 97}|
+-----+------------+

root
 |-- name: string (nullable = true)
 |-- prop: struct (nullable = true)
 |    |-- color: string (nullable = true)
 |    |-- age: long (nullable = true)



# column class

In [117]:
from pyspark.sql.functions import lit,col

column=lit("apple")
print(type(column))
column

<class 'pyspark.sql.column.Column'>


Column<'apple'>

In [118]:
data=[("nishant",353,'python'),("ketu",958,'sql')]
df=spark.createDataFrame(data,['name','salary','skill'])

# accessing column
df.select(col('name'),lit('salary')).show()

+-------+------+
|   name|salary|
+-------+------+
|nishant|salary|
|   ketu|salary|
+-------+------+



# when and otherwise

In [119]:
data=[("nishant",353,'python','M'),("ketu",958,'sql','F'),("suraj",853,'berojgar','')]
df=spark.createDataFrame(data,['name','salary','skill','gender'])
df.show()

from pyspark.sql.functions import when

df.select(df.name,df.salary,df.skill,when(df.gender=='M',"Male").when(df.gender=='F','Female')
          .otherwise("dont Know").alias("GENDER") ).show()

# if you not specify some value like then it will null, like if remove otherwise

+-------+------+--------+------+
|   name|salary|   skill|gender|
+-------+------+--------+------+
|nishant|   353|  python|     M|
|   ketu|   958|     sql|     F|
|  suraj|   853|berojgar|      |
+-------+------+--------+------+

+-------+------+--------+---------+
|   name|salary|   skill|   GENDER|
+-------+------+--------+---------+
|nishant|   353|  python|     Male|
|   ketu|   958|     sql|   Female|
|  suraj|   853|berojgar|dont Know|
+-------+------+--------+---------+



# Alias, asc, desc, cast, like

In [120]:
df.select(df.name,col('salary').alias("aukat")).show() # eg alias

df.sort(df.name.desc()).show() # desc or asc

+-------+-----+
|   name|aukat|
+-------+-----+
|nishant|  353|
|   ketu|  958|
|  suraj|  853|
+-------+-----+

+-------+------+--------+------+
|   name|salary|   skill|gender|
+-------+------+--------+------+
|  suraj|   853|berojgar|      |
|nishant|   353|  python|     M|
|   ketu|   958|     sql|     F|
+-------+------+--------+------+



In [121]:
df.printSchema()
df.select(df.name,df.salary.cast('int')).printSchema()  # cast

root
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- skill: string (nullable = true)
 |-- gender: string (nullable = true)

root
 |-- name: string (nullable = true)
 |-- salary: integer (nullable = true)



In [122]:
df.show()
df.select(df.name.like('s%')).show()  # like 
df.filter(df.name.like('s%')).show()

+-------+------+--------+------+
|   name|salary|   skill|gender|
+-------+------+--------+------+
|nishant|   353|  python|     M|
|   ketu|   958|     sql|     F|
|  suraj|   853|berojgar|      |
+-------+------+--------+------+

+------------+
|name LIKE s%|
+------------+
|       false|
|       false|
|        true|
+------------+

+-----+------+--------+------+
| name|salary|   skill|gender|
+-----+------+--------+------+
|suraj|   853|berojgar|      |
+-----+------+--------+------+



# Filter and where

In [123]:
df.filter(col('gender')=='M').show()

# we can write in String SQL expression
df.where("gender=='M'").show() 

+-------+------+------+------+
|   name|salary| skill|gender|
+-------+------+------+------+
|nishant|   353|python|     M|
+-------+------+------+------+

+-------+------+------+------+
|   name|salary| skill|gender|
+-------+------+------+------+
|nishant|   353|python|     M|
+-------+------+------+------+



# Distinct, Drop duplicate
distinct(): Removes duplicate rows based on all columns in the DataFrame.

df.distinct()

dropDuplicates(): Removes duplicate rows based on specific columns (if provided). otherwise work same

df.dropDuplicates(["column1", "column2"])

In [124]:
data=[("nishant",353,'python','M'),("ketu",958,'sql','F'),("suraj",853,'berojgar','M'),("ketu",958,'sql','F')]
df=spark.createDataFrame(data,['name','salary','skill','gender'])
df.show()

+-------+------+--------+------+
|   name|salary|   skill|gender|
+-------+------+--------+------+
|nishant|   353|  python|     M|
|   ketu|   958|     sql|     F|
|  suraj|   853|berojgar|     M|
|   ketu|   958|     sql|     F|
+-------+------+--------+------+



In [125]:
df.distinct().show()

df.dropDuplicates(["gender"]).show()

+-------+------+--------+------+
|   name|salary|   skill|gender|
+-------+------+--------+------+
|nishant|   353|  python|     M|
|   ketu|   958|     sql|     F|
|  suraj|   853|berojgar|     M|
+-------+------+--------+------+

+-------+------+------+------+
|   name|salary| skill|gender|
+-------+------+------+------+
|   ketu|   958|   sql|     F|
|nishant|   353|python|     M|
+-------+------+------+------+



# orderBy and sort
work same

In [126]:
df.sort(df.gender,df.salary.desc()).show()

df.orderBy(df.gender,df.salary.desc()).show()

+-------+------+--------+------+
|   name|salary|   skill|gender|
+-------+------+--------+------+
|   ketu|   958|     sql|     F|
|   ketu|   958|     sql|     F|
|  suraj|   853|berojgar|     M|
|nishant|   353|  python|     M|
+-------+------+--------+------+

+-------+------+--------+------+
|   name|salary|   skill|gender|
+-------+------+--------+------+
|   ketu|   958|     sql|     F|
|   ketu|   958|     sql|     F|
|  suraj|   853|berojgar|     M|
|nishant|   353|  python|     M|
+-------+------+--------+------+



# union and union all
here both will do same thing, in pyspark it doesnot remove duplicate by union

work on same schema df

In [127]:
data1=[("nishant",353,'python','M'),("ketu",958,'sql','F')]
df1=spark.createDataFrame(data1,['name','salary','skill','gender'])

data2=[("suraj",853,'berojgar','M'),("ketu",958,'sql','F')]
df2=spark.createDataFrame(data2,['name','salary','skill','gender'])

df=df1.union(df2)
df.show()


+-------+------+--------+------+
|   name|salary|   skill|gender|
+-------+------+--------+------+
|nishant|   353|  python|     M|
|   ketu|   958|     sql|     F|
|  suraj|   853|berojgar|     M|
|   ketu|   958|     sql|     F|
+-------+------+--------+------+



# groupBy, agg

In [128]:
# df.groupBy('gender').count().show()
df.groupBy('gender').max('salary').show()

from pyspark.sql.functions import max,count,max,min,sum
# df.groupBy('gender').agg(max('salary').alias('max_salary')).show()

df.groupBy('gender','skill').count().show()
df.groupBy('skill').agg(count('skill').alias('freq'),max('salary')).show()


+------+-----------+
|gender|max(salary)|
+------+-----------+
|     M|        853|
|     F|        958|
+------+-----------+

+------+--------+-----+
|gender|   skill|count|
+------+--------+-----+
|     M|  python|    1|
|     F|     sql|    2|
|     M|berojgar|    1|
+------+--------+-----+

+--------+----+-----------+
|   skill|freq|max(salary)|
+--------+----+-----------+
|  python|   1|        353|
|     sql|   2|        958|
|berojgar|   1|        853|
+--------+----+-----------+



# unionByName
* union: Combines DataFrames, matching columns by position.
* unionByName: Combines DataFrames, matching columns by name.
* it also work when schema is different, but you need to mention allowMissingColumns

In [129]:
data1=[("nishant",353,'python','M'),("ketu",958,'sql','F')]
df1=spark.createDataFrame(data1,['name','salary','skill','gender'])

data2=[("suraj",7,'berojgar'),("ketu",9,'sql')]
df2=spark.createDataFrame(data2,['name','age','skill'])

df=df1.unionByName(df2,allowMissingColumns=True)
df.show()

+-------+------+--------+------+----+
|   name|salary|   skill|gender| age|
+-------+------+--------+------+----+
|nishant|   353|  python|     M|NULL|
|   ketu|   958|     sql|     F|NULL|
|  suraj|  NULL|berojgar|  NULL|   7|
|   ketu|  NULL|     sql|  NULL|   9|
+-------+------+--------+------+----+



# select

In [130]:
# df.select('name','salary').show
# df.select('*').show()
df.select([col for col in df.columns]).show()

+-------+------+--------+------+----+
|   name|salary|   skill|gender| age|
+-------+------+--------+------+----+
|nishant|   353|  python|     M|NULL|
|   ketu|   958|     sql|     F|NULL|
|  suraj|  NULL|berojgar|  NULL|   7|
|   ketu|  NULL|     sql|  NULL|   9|
+-------+------+--------+------+----+



# join
* same as SQL join (left,right,full,inner,leftsemi,leftanti, self)
* left semi is same inner but only column present in left table will come
* left anti is opps of left semi, it get non-matching row from left df

In [131]:
data1=[("nishant",353,'python','2'),("ketu",958,'sql','3'),("ram",648,'sql','9')]
df1=spark.createDataFrame(data1,['name','salary','skill','dept_id'])

data2=[(1,'IT'),(2,"sales"),(3,"HR")]
df2=spark.createDataFrame(data2,['dept_id','dept_name'])

df1.join(df2).show()

df1.join(df2, df1.dept_id==df2.dept_id , 'inner').show()

df1.join(df2, df1.dept_id==df2.dept_id , 'left').show()

df1.join(df2, df1.dept_id==df2.dept_id , 'leftsemi').show()

df1.join(df2, df1.dept_id==df2.dept_id , 'leftanti').show()


+-------+------+------+-------+-------+---------+
|   name|salary| skill|dept_id|dept_id|dept_name|
+-------+------+------+-------+-------+---------+
|nishant|   353|python|      2|      1|       IT|
|nishant|   353|python|      2|      2|    sales|
|nishant|   353|python|      2|      3|       HR|
|   ketu|   958|   sql|      3|      1|       IT|
|   ketu|   958|   sql|      3|      2|    sales|
|   ketu|   958|   sql|      3|      3|       HR|
|    ram|   648|   sql|      9|      1|       IT|
|    ram|   648|   sql|      9|      2|    sales|
|    ram|   648|   sql|      9|      3|       HR|
+-------+------+------+-------+-------+---------+

+-------+------+------+-------+-------+---------+
|   name|salary| skill|dept_id|dept_id|dept_name|
+-------+------+------+-------+-------+---------+
|nishant|   353|python|      2|      2|    sales|
|   ketu|   958|   sql|      3|      3|       HR|
+-------+------+------+-------+-------+---------+

+-------+------+------+-------+-------+---------

# pivot

In [132]:
data=[("nishant",353,'python','M'),("ketu",358,'sql','F'),("meh",153,'python','M'),("le",353,'sql','M'),("be",458,'sql','F')]
df=spark.createDataFrame(data,['name','salary','skill','gender'])
df.show()
df.groupBy('skill','gender').count().show()
df.groupBy('skill').pivot('gender').count().show()

# if you want only some col for pivot
df.groupBy('skill').pivot('gender',['F']).count().show()

+-------+------+------+------+
|   name|salary| skill|gender|
+-------+------+------+------+
|nishant|   353|python|     M|
|   ketu|   358|   sql|     F|
|    meh|   153|python|     M|
|     le|   353|   sql|     M|
|     be|   458|   sql|     F|
+-------+------+------+------+

+------+------+-----+
| skill|gender|count|
+------+------+-----+
|python|     M|    2|
|   sql|     F|    2|
|   sql|     M|    1|
+------+------+-----+

+------+----+---+
| skill|   F|  M|
+------+----+---+
|   sql|   2|  1|
|python|NULL|  2|
+------+----+---+

+------+----+
| skill|   F|
+------+----+
|   sql|   2|
|python|NULL|
+------+----+



In [133]:
# To unpivot we use stack
# expr takes string and it run it as code
piv_df=df.groupBy('skill').pivot('gender').count()
df.groupBy('skill','gender').count().show()

from pyspark.sql.functions import expr
piv_df.select('skill',expr("stack(2,'M',M,'F',F) as (gender,count)")).show()


+------+------+-----+
| skill|gender|count|
+------+------+-----+
|python|     M|    2|
|   sql|     F|    2|
|   sql|     M|    1|
+------+------+-----+

+------+------+-----+
| skill|gender|count|
+------+------+-----+
|   sql|     M|    1|
|   sql|     F|    2|
|python|     M|    2|
|python|     F| NULL|
+------+------+-----+



# fill and fillna
both do same thing, full value at nulls

In [134]:
piv_df.show()
piv_df.fillna(0).show()

# if you want to do fillna in specific column pass a list of column as second parameter


+------+----+---+
| skill|   F|  M|
+------+----+---+
|   sql|   2|  1|
|python|NULL|  2|
+------+----+---+

+------+---+---+
| skill|  F|  M|
+------+---+---+
|   sql|  2|  1|
|python|  0|  2|
+------+---+---+



# sample

to take some sample set of row from a large data set

In [135]:
df=spark.range(start=1,end=21)
# df.show()

# first parameter take percent we want from data, and return random set of row
# below take 20 percent, but not exact

df.sample(fraction=0.2).show()

# if you pass some seed as then form same seed same value will come
df.sample(fraction=0.2,seed=999).show()

+---+
| id|
+---+
|  3|
|  8|
| 10|
| 14|
| 15|
| 17|
| 19|
+---+

+---+
| id|
+---+
|  3|
|  5|
|  8|
| 16|
+---+



# collect
In PySpark, collect() retrieves all rows of a DataFrame as a list of Row objects, bringing data from the distributed environment (cluster) to the local driver. It’s useful for small datasets or to preview data, but it should be avoided on large datasets as it can cause memory issues.

In [136]:
data=[("nishant",353,'python','M'),("ketu",358,'sql','F'),("meh",153,'python','M'),("le",353,'sql','M'),("be",458,'sql','F')]
df=spark.createDataFrame(data,['name','salary','skill','gender'])
mylist=df.collect()
print(mylist)
print(mylist[0][0])

[Row(name='nishant', salary=353, skill='python', gender='M'), Row(name='ketu', salary=358, skill='sql', gender='F'), Row(name='meh', salary=153, skill='python', gender='M'), Row(name='le', salary=353, skill='sql', gender='M'), Row(name='be', salary=458, skill='sql', gender='F')]
nishant


# transform

In [137]:
 # this if dataframe transform

from pyspark.sql.functions import upper

df.withColumn('name',upper('name')).show()

def myUpper(df):
    return df.withColumn('name',upper('name'))

df1=df.transform(myUpper)
df1.show()

+-------+------+------+------+
|   name|salary| skill|gender|
+-------+------+------+------+
|NISHANT|   353|python|     M|
|   KETU|   358|   sql|     F|
|    MEH|   153|python|     M|
|     LE|   353|   sql|     M|
|     BE|   458|   sql|     F|
+-------+------+------+------+

+-------+------+------+------+
|   name|salary| skill|gender|
+-------+------+------+------+
|NISHANT|   353|python|     M|
|   KETU|   358|   sql|     F|
|    MEH|   153|python|     M|
|     LE|   353|   sql|     M|
|     BE|   458|   sql|     F|
+-------+------+------+------+



* In PySpark, transform is a higher-order function that applies a custom transformation to each element of an array column, allowing for element-wise operations without needing to explode the array.
* transform is ideal for modifying array elements within each row, preserving array structure.

* df.transform(): A DataFrame method that applies a transformation function to the entire DataFrame, commonly used for chaining transformations.

* transform function: A PySpark function for transforming array columns element-wise within each row.

In [138]:
data=[("nishant",353,'python',[2,4,3,6]),("ketu",358,'sql',[1,6,5,3,5])]
df=spark.createDataFrame(data,['name','salary','skill','mark'])

from pyspark.sql.functions import transform,upper
df.select('name','salary','skill', transform('mark', lambda x: x*2).alias('extra')).show()

# from pyspark.sql.functions import expr
# df = df.withColumn("new_col", expr("transform(arr_col, x -> x * 2)"))

+-------+------+------+------------------+
|   name|salary| skill|             extra|
+-------+------+------+------------------+
|nishant|   353|python|     [4, 8, 6, 12]|
|   ketu|   358|   sql|[2, 12, 10, 6, 10]|
+-------+------+------+------------------+



In [139]:
def convertToUpper(x):
    return x*x
df.show()
df.select('name','salary',transform('mark',convertToUpper).alias('extra')).show()

+-------+------+------+---------------+
|   name|salary| skill|           mark|
+-------+------+------+---------------+
|nishant|   353|python|   [2, 4, 3, 6]|
|   ketu|   358|   sql|[1, 6, 5, 3, 5]|
+-------+------+------+---------------+

+-------+------+------------------+
|   name|salary|             extra|
+-------+------+------------------+
|nishant|   353|    [4, 16, 9, 36]|
|   ketu|   358|[1, 36, 25, 9, 25]|
+-------+------+------------------+



# createOrReplaceTempView
* createOrReplaceTempView creates a temporary SQL table from a DataFrame that you can query using SQL syntax, its availabe in same session.
* createOrReplaceGlobalTempView registers a DataFrame as a global temporary view that is accessible across all sessions within a Spark application.
* global temporary view by prefixing it with the global_temp database name:

In [140]:
data=[("nishant",353,'python','M'),("ketu",358,'sql','F'),("meh",153,'python','M'),("le",353,'sql','M'),("be",458,'sql','F')]
df=spark.createDataFrame(data,['name','salary','skill','gender'])

df.createOrReplaceTempView("temp_table")
spark.sql(" select * from temp_table ").show()

# we can also use but in specific session

# %sql
# SELECT * FROM temp_table



+-------+------+------+------+
|   name|salary| skill|gender|
+-------+------+------+------+
|nishant|   353|python|     M|
|   ketu|   358|   sql|     F|
|    meh|   153|python|     M|
|     le|   353|   sql|     M|
|     be|   458|   sql|     F|
+-------+------+------+------+



In [141]:
df.createOrReplaceGlobalTempView("global_temp_table")


# UDF

In [142]:
data=[("nishant",353,'python','M',78),("ketu",358,'sql','F',98),("meh",153,'python','M',65),("le",353,'sql','M',67),("be",458,'sql','F',24)]
df=spark.createDataFrame(data,['name','salary','skill','gender','bonus'])
df.show()

+-------+------+------+------+-----+
|   name|salary| skill|gender|bonus|
+-------+------+------+------+-----+
|nishant|   353|python|     M|   78|
|   ketu|   358|   sql|     F|   98|
|    meh|   153|python|     M|   65|
|     le|   353|   sql|     M|   67|
|     be|   458|   sql|     F|   24|
+-------+------+------+------+-----+



In [143]:
def total_pay(s,b):
    return s+b

from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

# first way to register
TotalPayment= udf(lambda s,b: total_pay(s,b),IntegerType())

# use of udf
df.withColumn("total_sal",TotalPayment(df.salary,df.bonus)).show()

+-------+------+------+------+-----+---------+
|   name|salary| skill|gender|bonus|total_sal|
+-------+------+------+------+-----+---------+
|nishant|   353|python|     M|   78|      431|
|   ketu|   358|   sql|     F|   98|      456|
|    meh|   153|python|     M|   65|      218|
|     le|   353|   sql|     M|   67|      420|
|     be|   458|   sql|     F|   24|      482|
+-------+------+------+------+-----+---------+



In [144]:
# second way to register udf
@udf(returnType=IntegerType())
def total_pay2(s,b):
    return s+b
    
# use of udf
df.withColumn("total_sal",total_pay2(df.salary,df.bonus)).show()


+-------+------+------+------+-----+---------+
|   name|salary| skill|gender|bonus|total_sal|
+-------+------+------+------+-----+---------+
|nishant|   353|python|     M|   78|      431|
|   ketu|   358|   sql|     F|   98|      456|
|    meh|   153|python|     M|   65|      218|
|     le|   353|   sql|     M|   67|      420|
|     be|   458|   sql|     F|   24|      482|
+-------+------+------+------+-----+---------+



In [145]:
# In PySpark, spark.udf.register is used to register a Python function as a User-Defined Function (UDF) 
# so it can be used in SQL queries or with DataFrame transformations.

from pyspark.sql.types import StringType

# Define the Python function
def convert_to_upper(text):
    return text.upper()

# Register the function as a UDF
spark.udf.register("toUpperUDF", convert_to_upper, StringType())

# Using in SQL
df.createOrReplaceTempView("temp_table")
spark.sql("SELECT toUpperUDF(name) FROM temp_table")

# Using with DataFrame
df.selectExpr("toUpperUDF(name) as upper_name").show()



+----------+
|upper_name|
+----------+
|   NISHANT|
|      KETU|
|       MEH|
|        LE|
|        BE|
+----------+



24/11/24 00:00:21 WARN SimpleFunctionRegistry: The function toupperudf replaced a previously registered function.


# RDD
# Resilient Distributed Dataset (RDD) in PySpark

In PySpark, **Resilient Distributed Dataset (RDD)** is the fundamental data structure, providing low-level APIs for distributed data processing. RDDs are fault-tolerant, distributed collections of objects that Spark can operate on in parallel.

## Key Features

- **Immutable**: Once created, RDDs can’t be changed; transformations create new RDDs.
- **Lazy Evaluation**: Transformations on RDDs (like `map`, `filter`) are not executed immediately but are triggered upon an action (like `collect`, `count`).
- **Fault-Tolerant**: Data can be recomputed if a partition is lost.
- **Distributed**: Data is split across clusters, enabling parallel processing.

## Common Operations

- **Transformations**: `map`, `filter`, `flatMap`, `reduceByKey`
- **Actions**: `collect`, `count`, `saveAsTextFile`



In [146]:
## Example Usage

# Creating an RDD
rdd = spark.sparkContext.parallelize([1, 2, 3, 4, 5])

# Applying a transformation
rdd2 = rdd.map(lambda x: x * 2)

# Triggering action
print(rdd2.collect())  # Output: [2, 4, 6, 8, 10]

[2, 4, 6, 8, 10]


In [147]:
# rdd to df

# Without specifying schema (infers column names automatically)
rdd = spark.sparkContext.parallelize([(1, "Alice"), (2, "Bob")])
df = rdd.toDF(["id", "name"])
df.show()

# With schema for structured RDD
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True)
])

df = rdd.toDF(schema)
df.show()


+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+



In [148]:
# other way of creating df from rdd
spark.createDataFrame(rdd).show()

+---+-----+
| _1|   _2|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+



# map and flat map

The `map` transformation applies a function to each element of an RDD and returns a new RDD containing the results. Each input element maps to exactly one output element.
* Each element in the original RDD is transformed to one corresponding element in the new RDD.
## Example
```python
rdd = spark.sparkContext.parallelize([1, 2, 3, 4])
rdd_mapped = rdd.map(lambda x: x * 2)
print(rdd_mapped.collect())  # Output: [2, 4, 6, 8]
```


The `flatMap` transformation in PySpark applies a function to each element of an RDD, creating an output of zero or more elements for each input element. It then flattens these elements into a single RDD, meaning the output is a single-level collection of all results.

## Example
In this example, each input string is split into multiple words, and all words are collected into a single RDD:

```python
rdd = spark.sparkContext.parallelize(["hello world", "foo bar"])
rdd_flatmapped = rdd.flatMap(lambda x: x.split(" "))
print(rdd_flatmapped.collect())  # Output: ['hello', 'world', 'foo', 'bar']
```
* flatMap is useful when each input element maps to multiple output elements and you want to combine them into a single collection.

Comparison with map
While map returns one element for each input element, flatMap can return multiple elements for each input and automatically flattens the result.


In [149]:
# Map
rdd = spark.sparkContext.parallelize(["hello","world", "foo","bar"])
rdd.collect

rdd.map(lambda x : x+" test").collect()

['hello test', 'world test', 'foo test', 'bar test']

In [150]:
# df to rdd

# Create a sample DataFrame
data = [("Alice", 30), ("Bob", 25), ("Charlie", 35)]
df = spark.createDataFrame(data, ["name", "age"])

# Convert DataFrame to RDD
rdd = df.rdd

# Show the RDD contents
print(rdd.collect())

rdd.map(lambda x : (x[0]+str(x[1]),'dead',)).collect()


[Row(name='Alice', age=30), Row(name='Bob', age=25), Row(name='Charlie', age=35)]


[('Alice30', 'dead'), ('Bob25', 'dead'), ('Charlie35', 'dead')]

In [151]:
# flatmap
rdd = spark.sparkContext.parallelize(["hello world", "foo bar"])
m=rdd.map(lambda x:x.split(' '))
print(m.collect())

fm=rdd.flatMap(lambda x:x.split(' '))
print(fm.collect())


[['hello', 'world'], ['foo', 'bar']]
['hello', 'world', 'foo', 'bar']


# partitionBy
* In PySpark, partitionBy is used to control the distribution of data across partitions based on specified columns, improving parallelism and efficiency, especially in operations like writing to files.

In [152]:
data=[("nishant",353,'python','M',78),("ketu",358,'sql','F',98),("meh",153,'python','M',65),("le",353,'sql','M',67),("be",458,'sql','F',24)]
df=spark.createDataFrame(data,['name','salary','skill','gender','bonus'])

In [153]:
# we can do partition on multiple folder by passing list of col
# this prititionBy will create a folders on which on column name on which its done, and column will be removed from it

df.write.parquet('./ partionFile',mode='overwrite',partitionBy='skill')

In [154]:
# example of partition on skill, col is not present, its now in form of folder
spark.read.parquet('./ partionFile/skill=python').show()

+-------+------+------+-----+
|   name|salary|gender|bonus|
+-------+------+------+-----+
|nishant|   353|     M|   78|
|    meh|   153|     M|   65|
+-------+------+------+-----+



In [155]:
# calling main folder will merge everything and folders columns will reapper
spark.read.parquet('./ partionFile').show()

+-------+------+------+-----+------+
|   name|salary|gender|bonus| skill|
+-------+------+------+-----+------+
|nishant|   353|     M|   78|python|
|   ketu|   358|     F|   98|   sql|
|    meh|   153|     M|   65|python|
|     le|   353|     M|   67|   sql|
|     be|   458|     F|   24|   sql|
+-------+------+------+-----+------+



# from_json()

- **Purpose**: Parses a JSON string column into a structured column with a specified schema.  

- **Syntax**:  
  ```python
  from pyspark.sql.functions import from_json
  df.withColumn("new_column", from_json(col("json_column"), schema))


In [156]:
data =[('ketu','{"hair":"black","eye":"brown"}')]
schema=['id','prop']
df=spark.createDataFrame(data,schema)
df.show(truncate=False)
df.printSchema()

+----+------------------------------+
|id  |prop                          |
+----+------------------------------+
|ketu|{"hair":"black","eye":"brown"}|
+----+------------------------------+

root
 |-- id: string (nullable = true)
 |-- prop: string (nullable = true)



In [157]:
from pyspark.sql.types import StructType, StructField, StringType,MapType
from pyspark.sql.functions import from_json

schema =MapType(StringType(),StringType())

df=df.withColumn("prop_map", from_json(col("prop"), schema))
df.show(truncate=False)
df.printSchema()


+----+------------------------------+-----------------------------+
|id  |prop                          |prop_map                     |
+----+------------------------------+-----------------------------+
|ketu|{"hair":"black","eye":"brown"}|{hair -> black, eye -> brown}|
+----+------------------------------+-----------------------------+

root
 |-- id: string (nullable = true)
 |-- prop: string (nullable = true)
 |-- prop_map: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)



In [158]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import from_json

schema =StructType([StructField('hair',StringType()),StructField('eye',StringType())])

df=df.withColumn("prop_struct", from_json(col("prop"), schema))
df.show(truncate=False)
df.printSchema()

+----+------------------------------+-----------------------------+--------------+
|id  |prop                          |prop_map                     |prop_struct   |
+----+------------------------------+-----------------------------+--------------+
|ketu|{"hair":"black","eye":"brown"}|{hair -> black, eye -> brown}|{black, brown}|
+----+------------------------------+-----------------------------+--------------+

root
 |-- id: string (nullable = true)
 |-- prop: string (nullable = true)
 |-- prop_map: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- prop_struct: struct (nullable = true)
 |    |-- hair: string (nullable = true)
 |    |-- eye: string (nullable = true)



# to_json()

- **Purpose**: Converts a structured column (e.g., StructType, MapType) into a JSON string column.  

- **Syntax**:  
  ```python
  from pyspark.sql.functions import to_json
  df.withColumn("json_column", to_json(col("struct_column")))


In [159]:
 from pyspark.sql.functions import to_json

df.withColumn("json_str_demo",to_json(col('prop_struct'))).show(truncate=False)

+----+------------------------------+-----------------------------+--------------+------------------------------+
|id  |prop                          |prop_map                     |prop_struct   |json_str_demo                 |
+----+------------------------------+-----------------------------+--------------+------------------------------+
|ketu|{"hair":"black","eye":"brown"}|{hair -> black, eye -> brown}|{black, brown}|{"hair":"black","eye":"brown"}|
+----+------------------------------+-----------------------------+--------------+------------------------------+



# json_tuple()

- **Purpose**: Extracts JSON object fields and returns them as columns.  
- **Syntax**:  
  ```python
  from pyspark.sql.functions import json_tuple
  df.select(json_tuple(col("json_column"), "key1", "key2")).show()


In [160]:
data =[('ketu','{"hair":"black","eye":"brown"}'),('aa','{"hair":"green","eye":"pink"}')]
df=spark.createDataFrame(data,['name','prop'])
df.show(truncate=False)

+----+------------------------------+
|name|prop                          |
+----+------------------------------+
|ketu|{"hair":"black","eye":"brown"}|
|aa  |{"hair":"green","eye":"pink"} |
+----+------------------------------+



In [161]:
from pyspark.sql.functions import json_tuple
df.select(df.name,json_tuple(df.prop,"hair","eye").alias("hair clr","eye clr")).show()

+----+--------+-------+
|name|hair clr|eye clr|
+----+--------+-------+
|ketu|   black|  brown|
|  aa|   green|   pink|
+----+--------+-------+



# get_json_object()

- **Purpose**: Extracts a specific JSON field from a JSON string column.  
- **Syntax**:  
  ```python
  from pyspark.sql.functions import get_json_object
  df.select(get_json_object(col("json_column"), "$.key")).show()

  "$.key" start with root of given json col"


In [162]:
data =[('ketu','{"hair":{"color":"red","size":{"len":10,"wid":20}},"eye":"brown"}')]
df=spark.createDataFrame(data,["name","prop"])
df.show(truncate=False)

+----+-----------------------------------------------------------------+
|name|prop                                                             |
+----+-----------------------------------------------------------------+
|ketu|{"hair":{"color":"red","size":{"len":10,"wid":20}},"eye":"brown"}|
+----+-----------------------------------------------------------------+



In [163]:
from pyspark.sql.functions import get_json_object
df.select(df.name,get_json_object(df.prop,"$.eye")).show()

+----+----------------------------+
|name|get_json_object(prop, $.eye)|
+----+----------------------------+
|ketu|                       brown|
+----+----------------------------+



In [164]:
df.select(df.name,get_json_object(df.prop,"$.hair.color")).show()
df.select(df.name,get_json_object(df.prop,"$.hair.size.len").alias('hair_len')).show()

+----+-----------------------------------+
|name|get_json_object(prop, $.hair.color)|
+----+-----------------------------------+
|ketu|                                red|
+----+-----------------------------------+

+----+--------+
|name|hair_len|
+----+--------+
|ketu|      10|
+----+--------+



# current_date(), date_format(), to_date()
#### current_date()
- Retrieves the current date in the default format `yyyy-MM-dd`

####  date_format()
- Formats a date column into a specific string format.

#### to_date()
-Converts a string to a date type.

In [179]:
df=spark.range(5)

from pyspark.sql.functions import current_date,date_format,to_date
df=df.withColumn("date",current_date())
df.show()

+---+----------+
| id|      date|
+---+----------+
|  0|2024-11-24|
|  1|2024-11-24|
|  2|2024-11-24|
|  3|2024-11-24|
|  4|2024-11-24|
+---+----------+



In [178]:
df.withColumn("formatted_date", date_format("date", "MM/dd/yyyy")).show()

+---+----------+--------------+
| id|      date|formatted_date|
+---+----------+--------------+
|  0|2024-11-24|    11/24/2024|
|  1|2024-11-24|    11/24/2024|
|  2|2024-11-24|    11/24/2024|
|  3|2024-11-24|    11/24/2024|
|  4|2024-11-24|    11/24/2024|
+---+----------+--------------+



In [183]:
df = spark.createDataFrame([("2024-11-14",)], ["date_string"])
df.printSchema()
df = df.withColumn("date", to_date("date_string","yyyy.dd.mm"))
df.show()
df.printSchema()


root
 |-- date_string: string (nullable = true)

+-----------+----+
|date_string|date|
+-----------+----+
| 2024-11-14|NULL|
+-----------+----+

root
 |-- date_string: string (nullable = true)
 |-- date: date (nullable = true)



# datediff
#### `datediff(start_date, end_date)`
- **`datediff()`** calculates the number of days between two dates. The result is a positive integer if `start_date` is before `end_date` and a negative integer if `start_date` is after `end_date`.

In [187]:
from pyspark.sql.functions import datediff, current_date,months_between

df = spark.createDataFrame([("2024-11-01",)], ["start_date"])
df = df.withColumn("end_date", current_date())

# Calculate the difference in days
df = df.withColumn("days_diff", datediff("end_date", "start_date"))
df.show()

#difference in month
df.withColumn("days_diff", months_between("end_date", "start_date")).show()

+----------+----------+---------+
|start_date|  end_date|days_diff|
+----------+----------+---------+
|2024-11-01|2024-11-24|       23|
+----------+----------+---------+

+----------+----------+----------+
|start_date|  end_date| days_diff|
+----------+----------+----------+
|2024-11-01|2024-11-24|0.74193548|
+----------+----------+----------+



In [193]:
# add months function
# similar function date_add
from pyspark.sql.functions import add_months,year

df.withColumn("added months", add_months("end_date", -12)).show()

# get year
df.withColumn("year", year("end_date")).show()

+----------+----------+---------+------------+
|start_date|  end_date|days_diff|added months|
+----------+----------+---------+------------+
|2024-11-01|2024-11-24|       23|  2023-11-24|
+----------+----------+---------+------------+

+----------+----------+---------+------------+----+
|start_date|  end_date|days_diff|added months|year|
+----------+----------+---------+------------+----+
|2024-11-01|2024-11-24|       23|  2023-11-24|2024|
+----------+----------+---------+------------+----+



# TimestampType

#### Overview
- **`TimestampType`** is a data type in PySpark used to store timestamp values, which includes both date and time information (e.g., `yyyy-MM-dd HH:mm:ss`).

#### Creation of `TimestampType`
- You can use **`current_timestamp()`** to get the current date and time in `TimestampType`.
- You can also explicitly cast a string to a timestamp using **`to_timestamp()`**.



In [202]:

from pyspark.sql.functions import current_timestamp, to_timestamp,minute
from pyspark.sql.types import TimestampType

# Example with current timestamp
df = spark.range(3).withColumn("current_time", current_timestamp())
df.printSchema()
df.show(truncate=False)



root
 |-- id: long (nullable = false)
 |-- current_time: timestamp (nullable = false)

+---+-------------------------+
|id |current_time             |
+---+-------------------------+
|0  |2024-11-24 00:37:01.58216|
|1  |2024-11-24 00:37:01.58216|
|2  |2024-11-24 00:37:01.58216|
+---+-------------------------+



In [203]:
# Example with string to timestamp conversion
df= spark.createDataFrame([("2024-11-24 14:30:00",)], ["timestamp_string"])
df = df.withColumn("timestamp", to_timestamp("timestamp_string"))
df.printSchema()
df.show()

root
 |-- timestamp_string: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)

+-------------------+-------------------+
|   timestamp_string|          timestamp|
+-------------------+-------------------+
|2024-11-24 14:30:00|2024-11-24 14:30:00|
+-------------------+-------------------+



In [204]:
df.withColumn(df.timestamp,minute(df.timestamp)).show()

PySparkTypeError: [NOT_ITERABLE] Column is not iterable.